# 01 - Data Loading and Exploratory Data Analysis

In this notebook, I start the project from the raw Steam interaction data.  
The goal of this step is to understand the structure of the dataset before moving into clustering, modeling, and dashboard development.

## 1. Import libraries and project paths

I first import the basic libraries used in the course and connect the notebook to the project folders.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.config import RAW_DATA_DIR, PROCESSED_DATA_DIR
from src.data_processing import load_steam_data, clean_steam_data, create_user_features

## 2. Load the raw dataset

The Steam dataset is stored as a raw CSV file under `data/raw`.  
The file does not include column names, so the loading function assigns the expected column names.

In [ ]:
raw_path = RAW_DATA_DIR / "steam-200k.csv"

df_raw = load_steam_data(raw_path)
df_raw.head()

## 3. Basic dataset overview

Before making any changes, I check the size of the raw dataset and the basic column information.

In [ ]:
df_raw.shape

In [ ]:
df_raw.info()

In [ ]:
df_raw.describe(include="all")

## 4. Missing values and duplicate rows

This step checks whether there are missing values or repeated rows in the raw data.

In [ ]:
df_raw.isna().sum()

In [ ]:
df_raw.duplicated().sum()

## 5. Clean the dataset

The raw file includes an unused final column.  
I remove this column and drop duplicate rows using the cleaning function in `src/data_processing.py`.

In [ ]:
df_clean = clean_steam_data(df_raw)

print("Raw shape:", df_raw.shape)
print("Clean shape:", df_clean.shape)

df_clean.head()

## 6. Action type analysis

The dataset contains two main action types: `purchase` and `play`.  
This distinction is important because purchase rows have value 1.0, while play rows represent gameplay hours.

In [ ]:
df_clean["action"].value_counts()

In [ ]:
df_clean["action"].value_counts(normalize=True)

In [ ]:
df_clean["action"].value_counts().plot(
    kind="bar",
    title="Action Type Counts",
    xlabel="Action Type",
    ylabel="Number of Interactions",
    grid=True
)

plt.show()

## 7. Game-level overview

Next, I check which games appear most often in the dataset.  
This gives a first idea about the most common games in the interaction records.

In [ ]:
top_games_by_interactions = df_clean["game"].value_counts().head(15)
top_games_by_interactions

In [ ]:
top_games_by_interactions.sort_values().plot(
    kind="barh",
    figsize=(8, 6),
    title="Top 15 Games by Number of Interactions",
    xlabel="Number of Interactions",
    ylabel="Game",
    grid=True
)

plt.show()

## 8. Playtime analysis

For playtime analysis, I only use rows where the action is `play`.  
Purchase rows are not treated as gameplay hours.

In [ ]:
play_df = df_clean[df_clean["action"] == "play"].copy()
play_df.head()

In [ ]:
play_df["hours"].describe()

In [ ]:
play_df["hours"].plot(
    kind="hist",
    bins=50,
    figsize=(8, 4),
    title="Distribution of Play Hours",
    xlabel="Hours",
    ylabel="Frequency",
    grid=True
)

plt.show()

The play hours distribution is expected to be right-skewed because some users spend very high amounts of time on a small number of games.
To make the distribution easier to read, I also look at a capped version.

In [ ]:
play_df[play_df["hours"] <= 100]["hours"].plot(
    kind="hist",
    bins=50,
    figsize=(8, 4),
    title="Distribution of Play Hours Capped at 100 Hours",
    xlabel="Hours",
    ylabel="Frequency",
    grid=True
)

plt.show()

## 9. Top games by total playtime

Interaction count and playtime are not the same.  
A game may appear many times, but another game may have much higher total playtime.

In [ ]:
top_games_by_hours = (
    play_df.groupby("game")["hours"]
    .sum()
    .sort_values(ascending=False)
    .head(15)
)

top_games_by_hours

In [ ]:
top_games_by_hours.sort_values().plot(
    kind="barh",
    figsize=(8, 6),
    title="Top 15 Games by Total Playtime",
    xlabel="Total Hours",
    ylabel="Game",
    grid=True
)

plt.show()

## 10. User-level feature table

The raw data is interaction-level.  
For clustering and modeling, I need user-level features.  
These features are created by grouping the raw data by `user_id`.

In [ ]:
user_features = create_user_features(df_clean)
user_features.head()

In [ ]:
user_features.shape

In [ ]:
user_features.describe()

## 11. User behavior distributions

I now inspect the main user-level features that will be used later for segmentation.

In [ ]:
user_features["total_hours"].plot(
    kind="hist",
    bins=50,
    figsize=(8, 4),
    title="Distribution of Total Gameplay Hours per User",
    xlabel="Total Hours",
    ylabel="Number of Users",
    grid=True
)

plt.show()

In [ ]:
user_features[user_features["total_hours"] <= 500]["total_hours"].plot(
    kind="hist",
    bins=50,
    figsize=(8, 4),
    title="Distribution of Total Gameplay Hours per User Capped at 500",
    xlabel="Total Hours",
    ylabel="Number of Users",
    grid=True
)

plt.show()

In [ ]:
user_features["unique_games"].plot(
    kind="hist",
    bins=50,
    figsize=(8, 4),
    title="Distribution of Unique Games per User",
    xlabel="Number of Unique Games",
    ylabel="Number of Users",
    grid=True
)

plt.show()

In [ ]:
user_features["purchase_ratio"].plot(
    kind="hist",
    bins=30,
    figsize=(8, 4),
    title="Distribution of Purchase Ratio",
    xlabel="Purchase Ratio",
    ylabel="Number of Users",
    grid=True
)

plt.show()

## 12. Correlation between user-level features

Finally, I check correlations between the user-level behavioral features.  
This helps understand which features move together before clustering and modeling.

In [ ]:
user_features.drop(columns=["user_id"]).corr()

In [ ]:
corr = user_features.drop(columns=["user_id"]).corr()

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr)

ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticklabels(corr.columns)

fig.colorbar(im, ax=ax)
ax.set_title("Correlation Matrix of User-Level Features")

plt.tight_layout()
plt.show()

## 13. Main observations from the initial EDA

At this stage, the main observations are:

- The dataset is interaction-level, so user-level feature engineering is needed.
- The data contains two action types: purchase and play.
- Purchase values should not be treated as gameplay hours.
- Playtime is highly skewed, with some users or games having very large values.
- The processed user-level table gives a better structure for clustering and modeling.

The next notebook will focus on feature engineering, scaling, K-Means clustering, and PCA visualization.